In [26]:
# --- setup ---
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import Callback
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import keras
from keras import layers, Model
from keras import ops as K
from keras import random as krandom   # <-- add this


In [22]:
import pandas as pd

# Example: load your dataset
df = pd.read_csv("/content/DMC2_S_CP2_52.csv")

# Drop last four columns
df = df.iloc[:, :-4]

print(df.shape)  # To verify

(42016, 52)


In [23]:
# ------------------------------------------
# 1) Load + split + scale (fit scaler on TRAIN only)
# ------------------------------------------

assert df.shape[1] == 52, f"Expected 52 features, got {df.shape[1]}"

X = df.values.astype("float32")

# If it’s time series and order matters, don’t shuffle:
X_train, X_tmp = train_test_split(X, test_size=0.2, shuffle=False)   # <<< CHANGE HERE (set shuffle=True for i.i.d. tabular)
X_val,  X_test = train_test_split(X_tmp, test_size=0.5, shuffle=False)

# Scale using train only (avoid leakage)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

input_dim = X_train_sc.shape[1]   # should be 52






In [24]:
# ------------------------------------------
# 2) VAE building blocks
# ------------------------------------------
class Sampling(layers.Layer):
    """Reparameterization trick: z = mu + exp(0.5*logvar) * eps"""
    def call(self, inputs):
        mu, logvar = inputs
        eps = random.normal(shape=K.shape(mu))          # Keras 3-friendly randomness
        return mu + K.exp(0.5 * logvar) * eps

def build_vae(input_dim=52,
              latent_dim=16,          # <<< CHANGE HERE (try 8/16/32)
              hidden_dims=(128, 64),  # <<< CHANGE HERE (capacity)
              dropout=0.1,            # <<< CHANGE HERE (0.0–0.2 typical)
              beta=1.0):              # <<< CHANGE HERE (β-VAE: 0.5–4.0)
    # ----- Encoder -----
    inputs = layers.Input(shape=(input_dim,))
    x = inputs
    for h in hidden_dims:
        x = layers.Dense(h, activation="elu")(x)
        if dropout and dropout > 0:
            x = layers.Dropout(dropout)(x)
    mu     = layers.Dense(latent_dim, name="z_mu")(x)
    logvar = layers.Dense(latent_dim, name="z_logvar")(x)
    z      = Sampling()([mu, logvar])
    encoder = Model(inputs, [mu, logvar, z], name="encoder")

    # ----- Decoder -----
    z_in = layers.Input(shape=(latent_dim,))
    y = z_in
    for h in reversed(hidden_dims):
        y = layers.Dense(h, activation="elu")(y)
    outputs = layers.Dense(input_dim, activation="linear")(y)
    decoder = Model(z_in, outputs, name="decoder")

    # ----- VAE end-to-end -----
    mu_out, logvar_out, z_out = encoder(inputs)
    recon = decoder(z_out)

    # --- Losses (use keras.ops, not tf.*) ---
    # Reconstruction loss: MSE, sum over features then mean over batch
    recon_loss_per_sample = K.sum(K.square(inputs - recon), axis=1)
    recon_loss = K.mean(recon_loss_per_sample)

    # KL divergence: sum over latent dims, mean over batch
    kl_per_sample = -0.5 * K.sum(1 + logvar_out - K.square(mu_out) - K.exp(logvar_out), axis=1)
    kl_loss = K.mean(kl_per_sample)

    total_loss = recon_loss + beta * kl_loss

    vae = Model(inputs, recon, name="vae")
    vae.add_loss(total_loss)
    vae.add_metric(recon_loss, name="recon_loss")
    vae.add_metric(kl_loss,    name="kl_loss")

    # Expose encoder/decoder for later use
    vae.encoder = encoder
    vae.decoder = decoder
    return vae

In [19]:
# ------------------------------------------
# 3) Optional: KL warm-up (gradually increase β)
# ------------------------------------------
class KLWarmUp(Callback):
    def __init__(self, vae_model, beta_start=0.0, beta_end=1.0, warmup_epochs=10):
        super().__init__()
        self.vae_model = vae_model
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.warmup_epochs = warmup_epochs

    def on_epoch_begin(self, epoch, logs=None):
        # linearly ramp beta from start to end
        if epoch < self.warmup_epochs:
            new_beta = self.beta_start + (self.beta_end - self.beta_start) * (epoch / max(1, self.warmup_epochs))
        else:
            new_beta = self.beta_end
        # rebuild the add_loss is complicated at runtime; simpler approach:
        # keep beta external (attribute) and multiply KL inside a custom training loop.
        # For simplicity, we'll rebuild the model once up front with final beta and
        # skip dynamic changes. If you really want dynamic beta, switch to a custom train_step.
        pass

In [25]:
latent_dim  = 16             # <<< CHANGE HERE
hidden_dims = (128, 64)      # <<< CHANGE HERE
dropout     = 0.1            # <<< CHANGE HERE
beta        = 1.0            # <<< CHANGE HERE

vae = build_vae(input_dim=input_dim,
                latent_dim=latent_dim,
                hidden_dims=hidden_dims,
                dropout=dropout,
                beta=beta)

vae.compile(optimizer=keras.optimizers.Adam(1e-3))
vae.summary()   # optional

history = vae.fit(
    X_train_sc, None,                        # targets are the inputs (autoencoder)
    validation_data=(X_val_sc, None),
    epochs=50,                               # <<< CHANGE HERE (keep same across comparisons)
    batch_size=256,
    verbose=1
)

NameError: Exception encountered when calling Sampling.call().

[1mCould not automatically infer the output shape / dtype of 'sampling_3' (of type Sampling). Either the `Sampling.call()` method is incorrect, or you need to implement the `Sampling.compute_output_spec() / compute_output_shape()` method. Error encountered:

name 'random' is not defined[0m

Arguments received by Sampling.call():
  • args=(['<KerasTensor shape=(None, 16), dtype=float32, sparse=False, ragged=False, name=keras_tensor_46>', '<KerasTensor shape=(None, 16), dtype=float32, sparse=False, ragged=False, name=keras_tensor_47>'],)
  • kwargs=<class 'inspect._empty'>

In [ ]:
# ------------------------------------------
# 5) Encode to reduced features (latent Z)
# ------------------------------------------
# Use the encoder's z output as reduced features
# Option 1: deterministic embeddings (use μ)
mu_train, _, _ = vae.encoder.predict(X_train_sc, batch_size=1024, verbose=0)
mu_val,   _, _ = vae.encoder.predict(X_val_sc,   batch_size=1024, verbose=0)
mu_test,  _, _ = vae.encoder.predict(X_test_sc,  batch_size=1024, verbose=0)

print("Original dim:", input_dim, "→ Reduced dim:", mu_train.shape[1])  # e.g., 52 → 16

# Save if you want
pd.DataFrame(mu_train).to_csv("vae_train_latent.csv", index=False)
pd.DataFrame(mu_val).to_csv("vae_val_latent.csv", index=False)
pd.DataFrame(mu_test).to_csv("vae_test_latent.csv", index=False)

# ----- quick reconstruction sanity check (optional) -----
z_val = vae.encoder.predict(X_val_sc, verbose=0)[2]
recon_val = vae.decoder.predict(z_val, verbose=0)
mse_val = np.mean((X_val_sc - recon_val)**2)
print("Validation reconstruction MSE:", mse_val)

In [33]:
# ===============================
# 0) Imports
# ===============================
import numpy as np
import pandas as pd
import keras
from keras import layers, Model
from keras import ops as K
from keras import random as krandom
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Keras:", keras.__version__)
print("TF:", tf.__version__)

# ===============================
# 1) Load your data (52 features only)
# ===============================
# If your CSV has extra columns (e.g., targets at the end), keep only the first 52:
df = pd.read_csv("/content/DMC2_S_CP2_52.csv").iloc[:, :52]
#df = pd.read_csv("/content/your_52_features.csv")  # <<< CHANGE HERE
assert df.shape[1] == 52, f"Expected 52 features, got {df.shape[1]}"
X = df.values.astype("float32")

# If time order matters, keep shuffle=False
X_train, X_tmp = train_test_split(X, test_size=0.2, shuffle=False)  # <<< CHANGE (shuffle=True) for i.i.d.
X_val,  X_test = train_test_split(X_tmp, test_size=0.5, shuffle=False)

# Scale with train only (avoid leakage)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

input_dim  = X_train_sc.shape[1]   # 52
latent_dim = 16                    # <<< CHANGE HERE (try 8/16/32)
hidden_dims = (128, 64)            # <<< CHANGE HERE
dropout = 0.10                     # <<< CHANGE HERE
beta = 1.0                         # <<< CHANGE HERE (β-VAE weight)

# ===============================
# 2) Encoder / Decoder (Keras-3 safe)
# ===============================
def build_encoder(input_dim=52, latent_dim=16, hidden_dims=(128, 64), dropout=0.0):
    inp = layers.Input(shape=(input_dim,))
    x = inp
    for h in hidden_dims:
        x = layers.Dense(h, activation="elu")(x)
        if dropout and dropout > 0:
            x = layers.Dropout(dropout)(x)
    mu     = layers.Dense(latent_dim, name="z_mu")(x)
    logvar = layers.Dense(latent_dim, name="z_logvar")(x)

    # Reparameterization INSIDE a Lambda using keras.ops + keras.random
    def reparam(args):
        mu_, logvar_ = args
        eps = krandom.normal(shape=K.shape(mu_))
        return mu_ + K.exp(0.5 * logvar_) * eps

    z = layers.Lambda(reparam, name="z")([mu, logvar])
    return Model(inp, [mu, logvar, z], name="encoder")

def build_decoder(output_dim=52, latent_dim=16, hidden_dims=(128, 64)):
    zin = layers.Input(shape=(latent_dim,))
    y = zin
    for h in reversed(hidden_dims):
        y = layers.Dense(h, activation="elu")(y)
    out = layers.Dense(output_dim, activation="linear")(y)
    return Model(zin, out, name="decoder")

encoder = build_encoder(input_dim, latent_dim, hidden_dims, dropout)
decoder = build_decoder(input_dim, latent_dim, hidden_dims)

# Optional: inspect
# encoder.summary()
# decoder.summary()

# ===============================
# 3) VAE with custom train_step (no add_loss)
# ===============================
class VAE(keras.Model):
    def __init__(self, encoder, decoder, beta=1.0, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.beta = beta
        self.total_loss_tracker = keras.metrics.Mean(name="total_loss")
        self.recon_loss_tracker = keras.metrics.Mean(name="recon_loss")
        self.kl_loss_tracker    = keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.recon_loss_tracker, self.kl_loss_tracker]

    def train_step(self, data):
        x = data
        with tf.GradientTape() as tape:
            mu, logvar, z = self.encoder(x, training=True)
            recon = self.decoder(z, training=True)

            # MSE reconstruction: sum over features, mean over batch
            recon_loss = tf.reduce_mean(tf.reduce_sum(tf.square(x - recon), axis=1))
            # KL divergence: sum over latent dims, mean over batch
            kl_loss = -0.5 * tf.reduce_mean(tf.reduce_sum(
                1 + logvar - tf.square(mu) - tf.exp(logvar), axis=1))
            total_loss = recon_loss + self.beta * kl_loss

        grads = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {"total_loss": self.total_loss_tracker.result(),
                "recon_loss": self.recon_loss_tracker.result(),
                "kl_loss": self.kl_loss_tracker.result()}

    def test_step(self, data):
        x = data
        mu, logvar, z = self.encoder(x, training=False)
        recon = self.decoder(z, training=False)
        recon_loss = tf.reduce_mean(tf.reduce_sum(tf.square(x - recon), axis=1))
        kl_loss = -0.5 * tf.reduce_mean(tf.reduce_sum(
            1 + logvar - tf.square(mu) - tf.exp(logvar), axis=1))
        total_loss = recon_loss + self.beta * kl_loss

        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {"total_loss": self.total_loss_tracker.result(),
                "recon_loss": self.recon_loss_tracker.result(),
                "kl_loss": self.kl_loss_tracker.result()}

vae = VAE(encoder, decoder, beta=beta)
vae.compile(optimizer=keras.optimizers.Adam(1e-3))

# ===============================
# 4) Train
# ===============================
history = vae.fit(
    X_train_sc,
    epochs=50,             # <<< keep consistent across comparisons
    batch_size=256,
    validation_data=(X_val_sc,),
    verbose=1
)

# ===============================
# 5) Get reduced features (μ as deterministic embeddings)
# ===============================
mu_train, _, _ = encoder.predict(X_train_sc, batch_size=1024, verbose=0)
mu_val,   _, _ = encoder.predict(X_val_sc,   batch_size=1024, verbose=0)
mu_test,  _, _ = encoder.predict(X_test_sc,  batch_size=1024, verbose=0)

print("Original → Reduced:", input_dim, "→", mu_train.shape[1])  # e.g., 52 → 16

# Optional: save latents
pd.DataFrame(mu_train).to_csv("vae_train_latent.csv", index=False)
pd.DataFrame(mu_val).to_csv("vae_val_latent.csv", index=False)
pd.DataFrame(mu_test).to_csv("vae_test_latent.csv", index=False)

# Quick reconstruction sanity check (optional)
z_val = encoder.predict(X_val_sc, verbose=0)[2]
recon_val = decoder.predict(z_val, verbose=0)
mse_val = float(np.mean((X_val_sc - recon_val)**2))
print("Validation reconstruction MSE:", mse_val)

Keras: 3.10.0
TF: 2.19.0
Epoch 1/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - kl_loss: 12.7549 - recon_loss: 42.4654 - total_loss: 55.2203 - val_kl_loss: 6.6135 - val_recon_loss: 20.3195 - val_total_loss: 26.9329
Epoch 2/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - kl_loss: 7.0317 - recon_loss: 14.8642 - total_loss: 21.8959 - val_kl_loss: 6.2896 - val_recon_loss: 18.6864 - val_total_loss: 24.9760
Epoch 3/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - kl_loss: 7.0454 - recon_loss: 10.6380 - total_loss: 17.6835 - val_kl_loss: 6.0608 - val_recon_loss: 18.9859 - val_total_loss: 25.0467
Epoch 4/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - kl_loss: 6.6867 - recon_loss: 9.2268 - total_loss: 15.9136 - val_kl_loss: 6.0227 - val_recon_loss: 18.2961 - val_total_loss: 24.3188
Epoch 5/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - kl_loss: 6.5590 - recon_loss: 8.3411 - total_loss: 14.9001 - val_kl_loss: 5.8565 - val_recon_loss: 18.7136 - val_total_loss: 24.5701
Epoch 6/50
132/132 ━━━━━━━━━━━━━━━━

# CNN+LSTM

In [41]:
"""
LSTM+CNN Autoencoder for Multivariate Time Series Dimensionality Reduction
===========================================================================

- Drops the **last 4 numeric features** from your dataset (works with 50–56 features).
- Builds a **hybrid CNN+LSTM autoencoder** to learn compact latent vectors (reduced features).
- Saves latent embeddings for train/val/test to CSV/NPY under `artifacts/`.
- Designed to run cleanly in **Colab** or a local notebook.

Fixes included:
- **All UpSampling1D layers are correctly called on tensors** (e.g., `x = UpSampling1D(...)(x)`).
- Clean functional Keras graph; no leftover layer objects passed positionally.

Dependencies: tensorflow>=2.10, numpy, pandas, scikit-learn
"""
from __future__ import annotations

import os
import math
import json
from dataclasses import dataclass
from typing import Tuple

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.preprocessing import StandardScaler

# -----------------------
# Configuration
# -----------------------
@dataclass
class Config:
    CSV_PATH: str = "/content/DMC2_S_CP2_52.csv"   # TODO: set to your CSV file path
    OUTPUT_DIR: str = "/content/artifacts"
    SEQ_LEN: int = 64                 # window length (must be divisible by 4)
    LATENT_DIM: int = 32              # size of reduced feature vector per window
    BATCH_SIZE: int = 128
    EPOCHS: int = 50
    VAL_RATIO: float = 0.1            # fraction of windows for validation
    TEST_RATIO: float = 0.1           # fraction of windows for test (at the end)
    LEARNING_RATE: float = 1e-3
    CONV_CHANNELS: Tuple[int, int, int] = (64, 64, 128)
    KERNEL_SIZE: int = 3
    DROPOUT: float = 0.1
    RANDOM_SEED: int = 42

CFG = Config()

# -----------------------
# Utils
# -----------------------

def set_seeds(seed: int = 42) -> None:
    np.random.seed(seed)
    tf.random.set_seed(seed)


def ensure_output_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def load_numeric_matrix(csv_path: str) -> Tuple[np.ndarray, list[str]]:
    """Load CSV, keep numeric columns; return (matrix, column_names).

    The matrix is shape (T_total, F_total_numeric). We'll **drop last 4** numeric cols later.
    """
    df = pd.read_csv(csv_path)
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(num_cols) < 5:
        raise ValueError(
            f"Need at least 5 numeric columns to drop the last 4. Found {len(num_cols)}."
        )
    X = df[num_cols].to_numpy(dtype=np.float32)
    return X, num_cols


def drop_last_four_features(X: np.ndarray, cols: list[str]) -> Tuple[np.ndarray, list[str]]:
    """Drop the last 4 features from the numeric matrix X and its column names."""
    X_reduced = X[:, :-4]
    cols_reduced = cols[:-4]
    return X_reduced, cols_reduced


def to_windows(X: np.ndarray, seq_len: int) -> np.ndarray:
    """Create overlapping sliding windows of length seq_len.

    Returns array of shape (N_windows, seq_len, n_features).
    """
    n_samples, n_features = X.shape
    if n_samples < seq_len:
        raise ValueError(
            f"Not enough rows ({n_samples}) for SEQ_LEN={seq_len}. Reduce SEQ_LEN or add data."
        )
    # Sliding windows with stride 1
    windows = np.lib.stride_tricks.sliding_window_view(X, (seq_len, n_features))
    windows = windows.reshape(-1, seq_len, n_features)
    return windows


def chrono_split(X: np.ndarray, val_ratio: float, test_ratio: float) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Chronological split: [0:train_end), [train_end:val_end), [val_end:]
    Returns (X_train, X_val, X_test)
    """
    n = len(X)
    if n < 10:
        raise ValueError("Too few windows for a stable split. Collect more data or reduce SEQ_LEN.")
    test_start = int(n * (1 - test_ratio))
    val_start = int(test_start * (1 - val_ratio))
    X_train = X[:val_start]
    X_val = X[val_start:test_start]
    X_test = X[test_start:]
    return X_train, X_val, X_test


def fit_scaler_on_train(X_train: np.ndarray) -> StandardScaler:
    scaler = StandardScaler()
    X2d = X_train.reshape(-1, X_train.shape[-1])
    scaler.fit(X2d)
    return scaler


def apply_scaler(X: np.ndarray, scaler: StandardScaler) -> np.ndarray:
    shape = X.shape
    X2d = X.reshape(-1, shape[-1])
    X2d = scaler.transform(X2d)
    return X2d.reshape(shape)


# -----------------------
# Model
# -----------------------

def build_cnn_lstm_autoencoder(seq_len: int, n_features: int, latent_dim: int) -> Tuple[Model, Model, Model]:
    """Build a CNN+LSTM autoencoder.

    Encoder: Conv1D -> BN -> Conv1D -> MaxPool -> Conv1D -> BN -> MaxPool -> LSTM -> Dense(latent)
    Decoder: RepeatVector(reduced_len) -> LSTM -> Conv1D -> UpSampling1D -> Conv1D -> UpSampling1D -> Conv1D (linear)
    """
    assert seq_len % 4 == 0, "SEQ_LEN must be divisible by 4 due to two MaxPooling1D(2) ops."

    inputs = layers.Input(shape=(seq_len, n_features))

    # --- Encoder ---
    x = layers.Conv1D(CFG.CONV_CHANNELS[0], CFG.KERNEL_SIZE, padding="same", activation=None)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(CFG.DROPOUT)(x)

    x = layers.Conv1D(CFG.CONV_CHANNELS[1], CFG.KERNEL_SIZE, padding="same", activation="relu")(x)
    x = layers.MaxPooling1D(pool_size=2)(x)  # seq_len / 2

    x = layers.Conv1D(CFG.CONV_CHANNELS[2], CFG.KERNEL_SIZE, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)  # seq_len / 4

    x = layers.LSTM(128, return_sequences=False)(x)
    latent = layers.Dense(latent_dim, name="latent")(x)

    # --- Decoder ---
    reduced_len = seq_len // 4
    x = layers.RepeatVector(reduced_len)(latent)
    x = layers.LSTM(128, return_sequences=True)(x)

    x = layers.Conv1D(CFG.CONV_CHANNELS[2], CFG.KERNEL_SIZE, padding="same", activation="relu")(x)
    x = layers.UpSampling1D(size=2)(x)  # back to seq_len / 2

    x = layers.Conv1D(CFG.CONV_CHANNELS[1], CFG.KERNEL_SIZE, padding="same", activation="relu")(x)
    x = layers.UpSampling1D(size=2)(x)  # back to seq_len

    outputs = layers.Conv1D(n_features, kernel_size=1, padding="same", activation="linear")(x)

    autoencoder = Model(inputs, outputs, name="cnn_lstm_autoencoder")
    encoder = Model(inputs, latent, name="encoder")

    # --- Standalone decoder (optional) ---
    latent_inp = layers.Input(shape=(latent_dim,), name="latent_input")
    y = layers.RepeatVector(reduced_len)(latent_inp)
    y = layers.LSTM(128, return_sequences=True)(y)
    y = layers.Conv1D(CFG.CONV_CHANNELS[2], CFG.KERNEL_SIZE, padding="same", activation="relu")(y)
    y = layers.UpSampling1D(size=2)(y)
    y = layers.Conv1D(CFG.CONV_CHANNELS[1], CFG.KERNEL_SIZE, padding="same", activation="relu")(y)
    y = layers.UpSampling1D(size=2)(y)
    y = layers.Conv1D(n_features, kernel_size=1, padding="same", activation="linear")(y)
    decoder = Model(latent_inp, y, name="decoder")

    opt = tf.keras.optimizers.Adam(learning_rate=CFG.LEARNING_RATE)
    autoencoder.compile(optimizer=opt, loss="mse")

    return autoencoder, encoder, decoder


# -----------------------
# Main training / export pipeline
# -----------------------

def main(cfg: Config = CFG) -> None:
    set_seeds(cfg.RANDOM_SEED)
    ensure_output_dir(cfg.OUTPUT_DIR)

    # 1) Load and prepare data
    X_full, numeric_cols = load_numeric_matrix(cfg.CSV_PATH)
    X, kept_cols = drop_last_four_features(X_full, numeric_cols)
    n_features = X.shape[1]
    print(f"Loaded data: total numeric cols={len(numeric_cols)}; after drop last 4 -> n_features={n_features}")

    if cfg.SEQ_LEN % 4 != 0:
        raise ValueError("CFG.SEQ_LEN must be divisible by 4 (due to pooling/upsampling).")

    windows = to_windows(X, cfg.SEQ_LEN)
    X_train, X_val, X_test = chrono_split(windows, cfg.VAL_RATIO, cfg.TEST_RATIO)

    # 2) Scale
    scaler = fit_scaler_on_train(X_train)
    X_train = apply_scaler(X_train, scaler)
    X_val = apply_scaler(X_val, scaler)
    X_test = apply_scaler(X_test, scaler)

    # 3) Build model
    autoencoder, encoder, decoder = build_cnn_lstm_autoencoder(cfg.SEQ_LEN, n_features, cfg.LATENT_DIM)
    autoencoder.summary()

    # 4) Train
    ckpt_path = os.path.join(cfg.OUTPUT_DIR, "autoencoder.keras")
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
        ModelCheckpoint(filepath=ckpt_path, monitor="val_loss", save_best_only=True),
    ]

    history = autoencoder.fit(
        X_train, X_train,
        validation_data=(X_val, X_val),
        epochs=cfg.EPOCHS,
        batch_size=cfg.BATCH_SIZE,
        shuffle=False,  # keep temporal order within windows
        callbacks=callbacks,
        verbose=1,
    )

    # 5) Export latent representations (reduced features)
    def export_latent(split: str, X_arr: np.ndarray) -> None:
        Z = encoder.predict(X_arr, batch_size=cfg.BATCH_SIZE, verbose=0)
        out_csv = os.path.join(cfg.OUTPUT_DIR, f"latent_{split}.csv")
        out_npy = os.path.join(cfg.OUTPUT_DIR, f"latent_{split}.npy")
        pd.DataFrame(Z).to_csv(out_csv, index=False)
        np.save(out_npy, Z)
        print(f"Saved {split} latent shape={Z.shape} -> {out_csv}")

    export_latent("train", X_train)
    export_latent("val", X_val)
    export_latent("test", X_test)

    # 6) Save scaler + kept col names for reproducibility
    meta = {
        "kept_columns": kept_cols,
        "seq_len": cfg.SEQ_LEN,
        "latent_dim": cfg.LATENT_DIM,
        "scaler_mean": scaler.mean_.tolist(),
        "scaler_scale": scaler.scale_.tolist(),
    }
    with open(os.path.join(cfg.OUTPUT_DIR, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)
    print(f"Wrote metadata to {os.path.join(cfg.OUTPUT_DIR, 'meta.json')}")


if __name__ == "__main__":
    main()


Loaded data: total numeric cols=56; after drop last 4 -> n_features=52


Model: "cnn_lstm_autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_16 (InputLayer)     │ (None, 64, 52)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_29 (Conv1D)              │ (None, 64, 64)         │        10,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 64, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_30 (Conv1D)              │ (None, 64, 64)         │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_10 (MaxPooling1D) │ (None, 32, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_31 (Conv1D)              │ (None, 32, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 32, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_11 (MaxPooling1D) │ (None, 16, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_10 (LSTM)                  │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent (Dense)                  │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_5 (RepeatVector)  │ (None, 16, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_11 (LSTM)                  │ (None, 16, 128)        │        82,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_32 (Conv1D)              │ (None, 16, 128)        │        49,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling1d_9 (UpSampling1D)  │ (None, 32, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_33 (Conv1D)              │ (None, 32, 64)         │        24,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling1d_10 (UpSampling1D) │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_34 (Conv1D)              │ (None, 64, 52)         │         3,380 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 343,316 (1.31 MB)

 Trainable params: 342,932 (1.31 MB)

 Non-trainable params: 384 (1.50 KB)

Epoch 1/50
266/266 ━━━━━━━━━━━━━━━━━━━━ 62s 202ms/step - loss: 1.9730 - val_loss: 0.4811 - learning_rate: 0.0010
Epoch 2/50
266/266 ━━━━━━━━━━━━━━━━━━━━ 53s 199ms/step - loss: 1.9921 - val_loss: 0.5078 - learning_rate: 0.0010
Epoch 3/50
266/266 ━━━━━━━━━━━━━━━━━━━━ 81s 198ms/step - loss: 1.9343 - val_loss: 0.5377 - learning_rate: 0.0010
Epoch 4/50
266/266 ━━━━━━━━━━━━━━━━━━━━ 80s 192ms/step - loss: 1.9299 - val_loss: 0.3864 - learning_rate: 0.0010
Epoch 5/50
266/266 ━━━━━━━━━━━━━━━━━━━━ 51s 192ms/step - loss: 2.0355 - val_loss: 0.4389 - learning_rate: 0.0010
Epoch 6/50
266/266 ━━━━━━━━━━━━━━━━━━━━ 81s 190ms/step - loss: 1.9766 - val_loss: 0.5754 - learning_rate: 0.0010
Epoch 7/50
266/266 ━━━━━━━━━━━━━━━━━━━━ 51s 192ms/step - loss: 1.8982 - val_loss: 0.5175 - learning_rate: 0.0010
Epoch 8/50
266/266 ━━━━━━━━━━━━━━━━━━━━ 81s 190ms/step - loss: 1.9693 - val_loss: 0.4814 - learning_rate: 0.0010
Epoch 9/50
266/266 ━━━━━━━━━━━━━━━━━━━━ 83s 193ms/step - loss: 2.0460 - val_loss: 0.5420 - learn